# 05 — Preprocessing Pipeline


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

import numpy as np
from neuro.config import PROCESSED_DIR
from neuro.bids import validate_bids
from neuro.features import get_schaefer_masker
from nilearn.image import smooth_img
import nibabel as nib
from neuro.mlflow_utils import start_run

with start_run("05_preprocessing"):
    report = validate_bids()
    row = report["runs"][report["runs"]["bold_exists"]].iloc[0]
    raw = nib.load(row["bold_path"])
    smoothed = smooth_img(raw, fwhm=6)
    masker = get_schaefer_masker(n_rois=100)
    ts_clean = masker.fit_transform(smoothed)
    np.save(PROCESSED_DIR / "example_preprocessed_ts.npy", ts_clean)
    mlflow.log_param("fwhm_mm", 6)
    print("Preprocessed shape:", ts_clean.shape)


In [ ]:

mid = raw.shape[3] // 2
z = raw.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(raw.get_fdata()[:, :, z, mid], cmap="gray")
axes[0].set_title("Raw BOLD (mid slice/vol)")
axes[1].imshow(smoothed.get_fdata()[:, :, z, mid], cmap="gray")
axes[1].set_title("Smoothed 6mm FWHM")
plt.show()
import pandas as pd
ts_df = pd.DataFrame(ts_clean, columns=[f"roi_{i}" for i in range(ts_clean.shape[1])])
sns.lineplot(data=ts_df.iloc[:, :5], dashes=False)
plt.title("Preprocessed ROI time series (first 5 ROIs)")
plt.xlabel("Volume")
plt.show()
